# Projet : la satisfaction des patients, de bout en bout

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. lire une documentation de donnees avant d'ecrire une ligne de code
2. transformer une demande floue en specification : target, maille, metric
3. reperer qu'une cle n'est pas celle que l'on croit, et le prouver
4. reperer que la cible est deja dans les colonnes
5. livrer une liste d'etablissements defendable, et savoir dire ce qu'elle vaut

## Prérequis

- avoir fini les TD 1 a 6
- rien d'autre : tout le reste est dans ce notebook

**Durée :** 2 heures

## Le contexte

L'Agence regionale de sante vous appelle. Elle dispose des resultats e-Satis, l'enquete nationale de satisfaction des patients hospitalises plus de 48 heures, pour 2017 et 2018. Sa question tient en une phrase : **quels etablissements faut-il aller voir en priorite ?**

Elle a les moyens d'en auditer une vingtaine. Pas davantage.

Vous avez deux fichiers bruts et un lexique. Personne ne vous dira si votre reponse est la bonne, parce que personne ne la connait. En revanche, chaque etape du chemin se verifie, et le notebook vous le dira.

## Le déroulé

| quand | quoi |
|---|---|
| 0:00 | etape 1 : lire la documentation, et cadrer |
| 0:20 | etape 2 : charger, et trouver la vraie maille |
| 0:45 | etape 3 : la cible, et les baselines |
| 1:05 | etape 4 : le modele, et les deux metriques |
| 1:30 | etape 5 : la liste, et ce qu'elle vaut |
| 1:45 | etape 6 : sans la cible, si le temps le permet |
| 2:00 | la recommandation, une page |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# lire un CSV administratif francais
t = pd.read_csv(chemin, sep=';', encoding='latin-1',
                dtype={'finess': str, 'finess_geo': str})

# compter les valeurs distinctes d une colonne
t['col'].nunique()

# joindre deux tables sur une cle
j = a.merge(b, on='cle', suffixes=('_17', '_18'))

# convertir en nombre ce qui doit l etre
pd.to_numeric(t['col'], errors='coerce')

# garder les lignes dont la valeur est dans une liste
t[t['col'].isin(['A', 'B', 'C', 'D'])]

# classer par probabilite decroissante
rang = np.argsort(-proba)
```

---

## Étape 1 : Lire la documentation, et cadrer

**20 minutes.** Vingt minutes sans ecrire une ligne de modele. C'est le meilleur investissement de la seance, et c'est celui que tout le monde saute.

### 1.1 Ouvrez le lexique

Le fichier `esatis_lexique.csv` dit ce que contient chaque colonne. Lisez-le en entier, il fait vingt lignes.

Deux colonnes meritent une lecture lente : `classement` et `finess`.

In [ ]:
# A FAIRE : afficher le lexique en entier, et le LIRE.
lexique = pd.read_csv(DONNEES / 'esatis_lexique.csv',
                      sep=';', encoding='latin-1')

pd.set_option('display.max_colwidth', 200)
...        # afficher les 20 lignes

**Indice 1.** `display(lexique)` ou simplement `lexique` en derniere ligne de la cellule. Avec `max_colwidth` a 200, la description de `classement` s'affiche en entier, et c'est elle qu'il faut lire.

In [ ]:
assert len(lexique) >= 20, f'attendu au moins 20 lignes, obtenu {len(lexique)}'
desc = ' '.join(lexique['LABEL'].astype(str))
assert '77.3' in desc or '77,3' in desc, (
    'le lexique devrait contenir les seuils du classement')
print('correct. Et maintenant la question qui compte :')
print()
print('  le lexique ecrit que la classe A commence a 77,3 et la D sous 70,7.')
print('  Le classement est donc une FONCTION du score de satisfaction.')
print('  Retenez-le : predire le classement a partir du score ne prouverait')
print('  rien du tout. Vous verrez a l etape 3 pourquoi cela compte.')

### 1.2 Ecrivez la specification

Avant de coder. La demande de l'ARS est **« quels etablissements aller voir »**, ce qui n'est pas un probleme de machine learning tant que vous n'avez pas decide trois choses.

Remplissez les trois chaines ci-dessous. Il n'y a pas de correction automatique : c'est votre contrat, et vous le relirez a la fin.

In [ ]:
# A FAIRE : votre specification, en francais, en une phrase chacune.
TARGET = '...'    # qu est-ce qu on predit exactement ?
MAILLE = '...'    # une ligne du tableau final, c est quoi ?
METRIC = '...'    # on jugera le resultat avec quoi, et pourquoi ?

for nom, valeur in [('target', TARGET), ('maille', MAILLE),
                    ('metric', METRIC)]:
    print(f'{nom:<8} {valeur}')

**Indice 1.** Trois indices, et aucun n'est la reponse. La target : l'ARS veut agir AVANT, donc elle ne veut pas savoir qui etait mauvais, elle veut savoir qui va le devenir. La maille : relisez la ligne `finess` du lexique. La metric : combien d'audits l'ARS peut-elle payer ?

---

## Étape 2 : Charger, et trouver la vraie maille

**25 minutes.** Deux fichiers administratifs francais, et une cle qui n'est pas celle que l'on croit.

### 2.1 Charger les deux annees

Ce sont de vrais fichiers de l'administration francaise. Separateur point-virgule, encodage latin-1, identifiants qui commencent par un zero.

Un `read_csv` sans arguments echoue, et c'est normal.

In [ ]:
# A FAIRE : charger les deux recueils.
#           Attention aux identifiants : ce sont des CHAINES, pas des nombres.
a17 = ...
a18 = ...

print('2017', a17.shape)
print('2018', a18.shape)

**Indice 1.** Le patron du memo. Si les finess perdent leur zero initial, c'est que pandas les a lus comme des entiers : forcez `dtype`.

**Indice 2.** `a17 = pd.read_csv(DONNEES / 'esatis_2017.csv', sep=';', encoding='latin-1', dtype={'finess': str, 'finess_geo': str})`

In [ ]:
assert a17.shape == (1157, 23), f'attendu (1157, 23), obtenu {a17.shape}'
assert a18.shape == (1153, 23), f'attendu (1153, 23), obtenu {a18.shape}'
assert a17['finess'].dtype == object, (
    'les finess doivent etre des chaines, sinon les zeros initiaux sautent')
print(f'correct. {len(a17):,} etablissements en 2017, {len(a18):,} en 2018.')

### 2.2 Laquelle des deux colonnes est la cle ?

Deux colonnes ressemblent a un identifiant : `finess` et `finess_geo`. **Pariez avant de lancer** : laquelle identifie une ligne ?

In [ ]:
# A FAIRE : pour CHAQUE annee, compter les valeurs distinctes des deux colonnes.
for an, t in [('2017', a17), ('2018', a18)]:
    ...        # afficher len(t), t['finess'].nunique(), t['finess_geo'].nunique()

**Indice 1.** `t['finess'].nunique()` contre `len(t)`. Si les deux ne sont pas egaux, la colonne n'identifie pas une ligne.

In [ ]:
assert a18['finess'].nunique() == 1001, (
    f"attendu 1 001 finess distincts en 2018, obtenu {a18['finess'].nunique()}")
assert a18['finess_geo'].nunique() == len(a18), (
    'finess_geo devrait identifier une ligne')
print('correct, et voici ce que cela veut dire :')
pires = a18['finess'].value_counts()
f = pires.index[0]
nom = a18.loc[a18['finess'] == f, 'rs_finess'].iloc[0]
print(f'  le finess {f} ({nom}) porte {pires.iloc[0]} sites.')
print('  finess est l entite JURIDIQUE, finess_geo est l etablissement.')
print('  Joindre sur finess ferait un produit cartesien, en silence.')

### 2.3 Joignez les deux annees

Avec la bonne cle. Si le nombre de lignes obtenu depasse celui de la plus petite des deux annees, c'est que vous avez pris l'autre.

In [ ]:
# A FAIRE : joindre 2017 et 2018, sur la bonne cle.
j = ...

print(f'{len(j):,} etablissements presents les deux annees')

**Indice 1.** `a17.merge(a18, on=..., suffixes=('_17', '_18'))`. Les suffixes sont obligatoires ici, parce que les deux annees ont exactement les memes noms de colonnes.

In [ ]:
assert len(j) == 1115, (
    f'attendu 1 115 etablissements, obtenu {len(j):,}. '
    'Si vous en avez beaucoup plus, vous avez joint sur finess.')
assert 'score_all_rea_ajust_17' in j.columns, (
    'les suffixes _17 et _18 manquent')
print(f'correct : {len(j):,} etablissements, et aucune ligne inventee.')
print(f'  {len(a17) - len(j)} presents seulement en 2017, '
      f'{len(a18) - len(j)} seulement en 2018.')

---

## Étape 3 : La cible, et les baselines

**20 minutes.** Construire la colonne a predire, puis mesurer ce que vaut de ne rien faire. Dans cet ordre.

### 3.1 Construisez la cible

Un etablissement **se degrade** si son classement 2018 est moins bon que celui de 2017. A est le meilleur, D le pire.

Attention : `DI` et `NR` ne sont pas des notes. Le lexique dit ce qu'ils sont. Un etablissement qui passe de C a DI n'est pas degrade, il est non mesure, et c'est une information differente.

In [ ]:
# A FAIRE : garder les etablissements classes A a D LES DEUX annees,
#           puis construire la cible binaire.
ORDRE = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

utiles = ...        # le sous-ensemble exploitable
r17 = ...           # le rang 2017, de 0 a 3
r18 = ...           # le rang 2018
y = ...             # 1 si degradation, 0 sinon

print(f'{len(utiles):,} etablissements exploitables')
print(f'taux de degradation : {y.mean():.4f}')

**Indice 1.** Pour filtrer, `isin(ORDRE)` sur les deux colonnes de classement. Pour le rang, `.map(ORDRE)`. La degradation, c'est un rang qui AUGMENTE.

**Indice 2.** `utiles = j[j['classement_17'].isin(ORDRE) & j['classement_18'].isin(ORDRE)].copy()`, puis `y = (r18 > r17).astype(int).values`

In [ ]:
assert len(utiles) == 596, (
    f'attendu 596 etablissements exploitables, obtenu {len(utiles):,}')
assert abs(y.mean() - 0.1376) < 0.002, (
    f'attendu un taux de 0,1376, obtenu {y.mean():.4f}')
print(f'correct. {int(y.sum())} degradations sur {len(y)}, soit {y.mean():.2%}.')
print()
print(f'RETENEZ ce chiffre : repondre toujours non donne deja '
      f'{1 - y.mean():.2%} d accuracy.')
print('Vous avez deja vu ce film au TD 3.')

### 3.2 Le baseline qui compte vraiment

« Toujours non » est le baseline betise. Il y en a un plus serieux, et c'est celui que l'ARS utilise deja sans le savoir : **regarder le classement de l'an dernier**.

Si un modele ne bat pas celui-la, il n'a aucune raison d'exister.

In [ ]:
# A FAIRE : deux baselines, sans aucun modele.
#   (1) 'toujours non'
#   (2) 'les plus mauvais de 2017 sont ceux qui vont se degrader'
#       -> classer par rang 2017 decroissant, prendre les 20 premiers
import numpy as np

base_toujours_non = ...

rang_naif = np.argsort(-r17.values)[:20]
rendement_naif = ...        # la part de degradations parmi ces 20

print(f'toujours non        : {base_toujours_non:.4f} d accuracy')
print(f'les 20 pires de 2017 : {rendement_naif:.1%} de rendement')

**Indice 1.** `base_toujours_non = 1 - y.mean()`. Pour le rendement naif, `y[rang_naif].mean()`.

In [ ]:
assert abs(base_toujours_non - 0.8624) < 0.005
print('correct.')
print()
print(f'Le baseline naif trouve {rendement_naif:.1%} de degradations')
print(f'parmi ses 20, contre {y.mean():.1%} au hasard.')
print()
print('C est LUI que votre modele doit battre, pas le hasard.')
print('Notez le chiffre : vous le comparerez a l etape 5.')

---

## Étape 4 : Le modele, et les deux metriques

**25 minutes.** Un modele, deux metriques, et le meme desaccord qu'au TD 3.

### 4.1 Entrainez, avec les seules donnees de 2017

**La regle absolue :** aucune colonne de 2018 dans les features. Le jour ou l'ARS utilisera ce modele, 2018 n'existera pas encore.

C'est la fuite temporelle du bloc E, et elle est facile a commettre ici parce que les colonnes se ressemblent a un suffixe pres.

In [ ]:
# A FAIRE : construire X avec les colonnes 2017 SEULEMENT, puis entrainer.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

DIMENSIONS = ['accueil', 'PECinf', 'PECmed', 'chambre', 'repas', 'sortie']
feats = [f'score_{d}_rea_ajust_17' for d in DIMENSIONS]
feats += ['score_all_rea_ajust_17', 'nb_rep_score_all_rea_ajust_17']

X = ...        # les features, converties en nombres
X['region'] = utiles['region_17'].astype('category')
X['classement_17'] = r17.values

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

modele = ...
proba = ...        # la probabilite de degradation sur le test

**Indice 1.** `X = utiles[feats].apply(pd.to_numeric, errors='coerce')`. Pour le modele, `HistGradientBoostingClassifier(random_state=42, categorical_features=['region'])`.

**Indice 2.** `proba = modele.predict_proba(Xte)[:, 1]`

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
acc = accuracy_score(yte, modele.predict(Xte))
auc = roc_auc_score(yte, proba)
base = 1 - yte.mean()
assert auc > 0.75, f'AUC attendue au-dessus de 0,75, obtenue {auc:.4f}'
assert not any('_18' in c for c in X.columns), (
    'une colonne de 2018 s est glissee dans les features : c est la fuite')
print(f'  toujours non      accuracy {base:.4f}   AUC 0.5000')
print(f'  gradient boosting accuracy {acc:.4f}   AUC {auc:.4f}')
print()
verdict = 'PERD' if acc < base else 'gagne'
print(f'En accuracy, votre modele {verdict} contre repondre toujours non.')
print(f'Et pourtant son AUC vaut {auc:.4f}, tres au-dessus du hasard.')
print('Les deux sont vraies. C est la lecon du TD 3, sur d autres donnees.')

---

## Étape 5 : La liste, et ce qu'elle vaut

**15 minutes.** L'ARS ne veut pas une AUC. Elle veut vingt noms.

### 5.1 Les vingt etablissements

Classez le test set par probabilite decroissante, prenez les vingt premiers, et mesurez le rendement. Puis comparez-le au baseline naif de l'etape 3.2.

In [ ]:
# A FAIRE : la liste des 20, et son rendement.
rang = np.argsort(-proba)[:20]
rendement = ...

print(f'rendement du modele : {rendement:.1%}')
print(f'rendement du naif   : {rendement_naif:.1%}')
print(f'au hasard           : {yte.mean():.1%}')

**Indice 1.** `rendement = yte[rang].mean()`. Le `rendement_naif` a ete calcule a l'etape 3.2, sur l'ensemble complet : la comparaison est donc indicative et pas parfaitement appariee. Le dire est plus honnete que de le cacher.

In [ ]:
assert rendement > 2 * yte.mean(), (
    f'rendement {rendement:.1%} attendu bien au-dessus du hasard '
    f'{yte.mean():.1%}')
print(f'correct. {int(yte[rang].sum())} degradations parmi vos 20,')
print(f'soit {rendement / yte.mean():.1f} fois le hasard.')
print()
print('VOILA le livrable. Pas l AUC, pas l accuracy : vingt noms, et un')
print('rendement que vous pouvez defendre devant quelqu un qui conteste')
print('vos hypotheses.')

### 5.2 Et ce que la liste ne dit pas

Avant de la livrer, trois limites a ecrire. Elles ne sont pas des precautions de style : si vous ne les ecrivez pas, quelqu'un les decouvrira a votre place, et plus tard.

In [ ]:
# A FAIRE : trois limites de votre liste, en une phrase chacune.
#           Aucune correction automatique. C est votre jugement.
LIMITES = [
    '...',
    '...',
    '...',
]
for i, l in enumerate(LIMITES, 1):
    print(f'{i}. {l}')

**Indice 1.** Trois pistes, et il y en a d'autres. Combien d'etablissements avez-vous ecartes a l etape 3, et que deviennent-ils ? Sur combien de lignes le test set repose-t-il vraiment ? Et une degradation de A vers B merite-t-elle le meme audit qu'une degradation de C vers D ?

---

## Étape 6 : Sans la cible, si le temps le permet

**15 minutes.** Les six dimensions de satisfaction racontent-elles six choses differentes ? Le bloc F, applique a un autre jeu.

### 6.1 Six dimensions, ou une seule ?

L'enquete mesure six choses : accueil, prise en charge paramedicale, prise en charge medicale, chambre, repas, sortie. L'ARS aimerait savoir si un etablissement peut etre bon sur l'une et mauvais sur l'autre, ou si tout monte et descend ensemble.

C'est une question sans label, et elle se repond avec une PCA.

In [ ]:
# A FAIRE : la correlation moyenne entre dimensions, puis la PCA.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

dims = utiles[[f'score_{d}_rea_ajust_17' for d in DIMENSIONS]]
dims = dims.apply(pd.to_numeric, errors='coerce').dropna()
Xs = StandardScaler().fit_transform(dims)

cumul = ...        # la variance expliquee cumulee
print(f'1 composante : {cumul[0]:.1%}')
print(f'2 composantes : {cumul[1]:.1%}')

**Indice 1.** `pca = PCA().fit(Xs)` puis `cumul = np.cumsum(pca.explained_variance_ratio_)`

In [ ]:
assert abs(cumul[0] - 0.642) < 0.02, (
    f'attendu environ 64 % sur la premiere composante, obtenu {cumul[0]:.1%}')
print(f'correct. Une seule composante porte {cumul[0]:.1%} de la variance.')
print()
print('Autrement dit : un etablissement est bon PARTOUT ou mauvais partout.')
print('Les six dimensions n en font presque qu une.')
print()
print('COMPAREZ au TD 6 : sur les sejours hospitaliers, il fallait SEPT')
print('composantes sur huit pour atteindre 90 %. Meme technique, resultat')
print('oppose. Ce n est pas la methode qui decide, ce sont les donnees.')

## La recommandation

Une page, et elle se relit a voix haute en deux minutes. Six points, dans cet ordre :

1. **La question**, reformulee : ce que l'ARS vous a demande, et ce que vous avez decide d'en faire.
2. **La specification** de l'etape 1.2, recopiee telle quelle.
3. **Les vingt etablissements**, avec leur probabilite.
4. **Ce que vaut la liste** : le rendement, contre le hasard et contre le baseline naif.
5. **Les trois limites** de l'etape 5.2.
6. **Ce que vous feriez avec une semaine de plus**, en une phrase.

Ce qui sera lu en premier, c'est le point 4. Ce qui vous evitera un probleme, c'est le point 5.

## Ce que cette étape a établi

- Vingt minutes de lecture de documentation ont evite deux pieges que le code n'aurait jamais signales.
- La cle n'etait pas celle que l'on croyait : finess est juridique, finess_geo est l etablissement, et l AP-HP porte 21 sites sous un seul finess.
- La cible etait deja dans les colonnes, et le lexique le disait.
- L accuracy perd contre « toujours non », l AUC vaut 0,84, et les deux sont vraies en meme temps.
- Le livrable n est pas une metrique, c est une liste de vingt noms avec son rendement et ses limites ecrites.

## Les erreurs classiques

- **Joindre sur `finess`** produit cartesien silencieux, 1 920 lignes au lieu de 1 115.
- **Predire le classement depuis le score** le lexique donne les seuils : c est une fonction, pas une prediction.
- **Glisser une colonne `_18` dans les features** fuite temporelle, et le score monte.
- **Livrer l AUC a l ARS** elle ne peut pas auditer une AUC.

## Pour aller plus loin, si vous avez fini en avance

- Refaire l etape 4 avec une colonne de 2018 dans les features, et mesurer de combien l AUC monte. C est votre propre fuite, fabriquee expres.
- Reprendre la cible : au lieu de « se degrade », predire « passe en D ». Le taux de base change, et le livrable aussi.
- Telecharger le recueil 2019 sur data.gouv.fr et verifier si votre liste de 2018 avait vu juste. C est le seul vrai test.